# Gold Skill Demand Mart

**Purpose**: Dashboard-ready skill demand trends by sector, role, and location.

**Target Table**: `workspace.gold.gold_skill_demand`

**Grain**: One row per date per skill per sector/role/location combination

**Key Metrics**:
- Skill mention counts
- Unique jobs requiring each skill
- Average confidence scores
- Period-over-period demand changes

**Data Sources**:
- `workspace.warehouse.bridge_job_skill`
- `workspace.warehouse.fact_job_postings`
- `workspace.warehouse.dim_skill`

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_skill_demand
USING DELTA
COMMENT 'Skill demand trends by sector, role, and location'
AS

WITH skill_job_facts AS (
  SELECT
    f.posting_date_sk AS demand_date_sk,
    f.sector_sk,
    f.role_sk,
    f.location_sk,
    bjs.skill_sk,
    bjs.job_sk,
    bjs.confidence,
    f.active_flag
  FROM workspace.warehouse.bridge_job_skill bjs
  JOIN workspace.warehouse.fact_job_postings f 
    ON bjs.job_sk = f.job_sk
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
    AND bjs.is_current = TRUE
    AND f.active_flag = TRUE
),

base_metrics AS (
  SELECT
    sjf.demand_date_sk,
    sjf.sector_sk,
    sjf.role_sk,
    sjf.location_sk,
    sjf.skill_sk,
    
    -- MEASURES
    COUNT(*) AS mentions_count,
    COUNT(DISTINCT sjf.job_sk) AS unique_jobs_count,
    ROUND(AVG(sjf.confidence), 4) AS avg_confidence,
    ROUND(MIN(sjf.confidence), 4) AS min_confidence,
    ROUND(MAX(sjf.confidence), 4) AS max_confidence
    
  FROM skill_job_facts sjf
  GROUP BY sjf.demand_date_sk, sjf.sector_sk, sjf.role_sk, sjf.location_sk, sjf.skill_sk
),

-- Rollup: By Sector and Skill (all roles, all locations)
sector_skill_agg AS (
  SELECT
    demand_date_sk,
    sector_sk,
    CAST(NULL AS BIGINT) AS role_sk,
    CAST(NULL AS BIGINT) AS location_sk,
    skill_sk,
    SUM(mentions_count) AS mentions_count,
    SUM(unique_jobs_count) AS unique_jobs_count,
    ROUND(AVG(avg_confidence), 4) AS avg_confidence
  FROM base_metrics
  GROUP BY demand_date_sk, sector_sk, skill_sk
),

-- Rollup: By Role and Skill (all sectors, all locations)
role_skill_agg AS (
  SELECT
    demand_date_sk,
    CAST(-1 AS BIGINT) AS sector_sk,
    role_sk,
    CAST(NULL AS BIGINT) AS location_sk,
    skill_sk,
    SUM(mentions_count) AS mentions_count,
    SUM(unique_jobs_count) AS unique_jobs_count,
    ROUND(AVG(avg_confidence), 4) AS avg_confidence
  FROM base_metrics
  GROUP BY demand_date_sk, role_sk, skill_sk
),

-- Rollup: By Location and Skill (all sectors, all roles)
location_skill_agg AS (
  SELECT
    demand_date_sk,
    CAST(-1 AS BIGINT) AS sector_sk,
    CAST(NULL AS BIGINT) AS role_sk,
    location_sk,
    skill_sk,
    SUM(mentions_count) AS mentions_count,
    SUM(unique_jobs_count) AS unique_jobs_count,
    ROUND(AVG(avg_confidence), 4) AS avg_confidence
  FROM base_metrics
  GROUP BY demand_date_sk, location_sk, skill_sk
),

-- Combine all aggregation levels
combined_agg AS (
  SELECT demand_date_sk, sector_sk, role_sk, location_sk, skill_sk, 
         mentions_count, unique_jobs_count, avg_confidence
  FROM sector_skill_agg
  
  UNION ALL
  
  SELECT demand_date_sk, sector_sk, role_sk, location_sk, skill_sk, 
         mentions_count, unique_jobs_count, avg_confidence
  FROM role_skill_agg
  
  UNION ALL
  
  SELECT demand_date_sk, sector_sk, role_sk, location_sk, skill_sk, 
         mentions_count, unique_jobs_count, avg_confidence
  FROM location_skill_agg
),

-- Add temporal metrics
temporal_enriched AS (
  SELECT
    ca.*,
    
    -- Prior period comparison (7 days ago)
    LAG(ca.mentions_count, 7) OVER (
      PARTITION BY ca.sector_sk, ca.role_sk, ca.location_sk, ca.skill_sk
      ORDER BY ca.demand_date_sk
    ) AS prev_week_mentions,
    
    -- Prior period comparison (30 days ago)
    LAG(ca.mentions_count, 30) OVER (
      PARTITION BY ca.sector_sk, ca.role_sk, ca.location_sk, ca.skill_sk
      ORDER BY ca.demand_date_sk
    ) AS prev_month_mentions,
    
    -- 7-day rolling average
    AVG(ca.mentions_count) OVER (
      PARTITION BY ca.sector_sk, ca.role_sk, ca.location_sk, ca.skill_sk
      ORDER BY ca.demand_date_sk
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_avg_mentions,
    
    -- 30-day rolling total
    SUM(ca.mentions_count) OVER (
      PARTITION BY ca.sector_sk, ca.role_sk, ca.location_sk, ca.skill_sk
      ORDER BY ca.demand_date_sk
      ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ) AS rolling_30day_total_mentions
    
  FROM combined_agg ca
)

SELECT
  -- DIMENSIONS
  te.demand_date_sk,
  te.sector_sk,
  te.role_sk,
  te.location_sk,
  te.skill_sk,
  
  -- MEASURES
  te.mentions_count,
  te.unique_jobs_count,
  te.avg_confidence,
  
  -- TEMPORAL METRICS
  te.rolling_7day_avg_mentions,
  te.rolling_30day_total_mentions,
  
  -- DERIVED: Change vs previous periods
  CAST((te.mentions_count - COALESCE(te.prev_week_mentions, 0)) AS BIGINT) AS delta_vs_prev_week,
  CAST((te.mentions_count - COALESCE(te.prev_month_mentions, 0)) AS BIGINT) AS delta_vs_prev_month,
  
  -- DERIVED: Percent change
  CASE 
    WHEN te.prev_week_mentions > 0 
    THEN CAST((te.mentions_count - te.prev_week_mentions) AS DECIMAL(10,4)) / te.prev_week_mentions
    ELSE NULL 
  END AS pct_change_vs_prev_week,
  
  -- METADATA
  CURRENT_TIMESTAMP() AS created_at,
  UUID() AS batch_id
  
FROM temporal_enriched te
ORDER BY te.demand_date_sk DESC, te.mentions_count DESC;

In [0]:
%sql
-- Validation: Top skills by demand
SELECT
  s.skill_name,
  s.skill_category,
  SUM(gsd.mentions_count) AS total_mentions,
  SUM(gsd.unique_jobs_count) AS total_unique_jobs,
  ROUND(AVG(gsd.avg_confidence), 4) AS overall_avg_confidence,
  COUNT(DISTINCT gsd.demand_date_sk) AS days_with_demand
FROM workspace.gold.gold_skill_demand gsd
JOIN workspace.warehouse.dim_skill s ON gsd.skill_sk = s.skill_sk
WHERE gsd.sector_sk != -1  -- Focus on sector-level aggregations
  AND gsd.role_sk IS NULL
  AND gsd.location_sk IS NULL
GROUP BY s.skill_name, s.skill_category
ORDER BY total_mentions DESC
LIMIT 20;